In [1]:
import hydra
from omegaconf import OmegaConf, DictConfig
from typing import Mapping
from orchestrator.config import MainConfig
import json

In [2]:
cfg_dir = "/home/kylian/work/NEEDLE/orchestrator/conf"

hydra.initialize_config_dir(
    config_dir=cfg_dir,
    version_base=None,
)

hydra.initialize_config_dir()

In [17]:
cfg_dict = hydra.compose(config_name="config")
cfg_defaults: DictConfig = OmegaConf.structured(MainConfig)
cfg: MainConfig = OmegaConf.merge(cfg_defaults, cfg_dict)  # type: ignore

In [4]:
print(json.dumps(OmegaConf.to_container(cfg), indent=4))

{
    "estimators": {
        "model_A": {
            "datamodule": "padded",
            "datamodule_override": null,
            "dataset": "fair_universe",
            "dataset_override": {
                "paths": "",
                "features_columns": null,
                "labels_columns": null,
                "format": "automatic",
                "dak_reader_kwargs": {},
                "max_number_events": -1
            },
            "model": "mock_transformer",
            "model_override": null,
            "trainer": "default",
            "trainer_override": null,
            "expands": {
                "ensembles": {
                    "num_ensembles": 5,
                    "aggregation_method": "mean"
                },
                "systematics": {
                    "up_qcd": {
                        "datamodule": null,
                        "datamodule_override": null,
                        "dataset": "fair_universe_local",
                        "da

In [38]:
from typing import Mapping, Literal
import hydra
from omegaconf import OmegaConf, DictConfig

DEFAULT_GROUPS: Mapping[str, str] = {
    "dataset": "datasets",
    "datamodule": "datamodules",
    "model": "models",
    "trainer": "trainers",
}


def _load_group(group: str, name: str) -> DictConfig:
    cfg = hydra.compose(overrides=[f"+{group}={name}"])
    return cfg[group]


def resolve_defaults(cfg: DictConfig, node_type: Literal["estimators", "systematics"] = "estimators") -> DictConfig:
    estimators: DictConfig = cfg[node_type]

    for _, est_cfg in estimators.items():
        for field, group in DEFAULT_GROUPS.items():
            group_member = est_cfg.get(field)

            if group_member is None:
                continue
            
            override_key = f"{field}_override"
            group_cfg =  _load_group(group, group_member)
            base_cfg = est_cfg.get(override_key)

            if base_cfg:
                est_cfg[override_key] = OmegaConf.merge(base_cfg, group_cfg)
            else:
                est_cfg[override_key] = group_cfg

    return cfg

cfg_resolved = resolve_defaults(cfg)
cfg_resolved.estimators.model_A

{'datamodule': 'padded', 'datamodule_override': {'_target_': 'ml.lightning.data.padded_datamodule.PaddedDataModule', 'multiprocessing_type': 'torch', 'batch_size': 1024}, 'dataset': 'fair_universe', 'dataset_override': {'paths': '/home/kylian/work/NEEDLE/FAIR_Universe_HiggsML_data/FAIR_Universe_HiggsML_data.parquet', 'features_columns': ['PRI_lep_pt'], 'labels_columns': ['PRI_n_jets'], 'format': 'automatic', 'dak_reader_kwargs': {}, 'max_number_events': 100000}, 'model': 'mock_transformer', 'model_override': {'_target_': 'ml.lightning.models.mock_transformer.MockTransformerModule', 'factor': 0.1, 'patience': 10, 'init_lr': 0.001}, 'trainer': 'default', 'trainer_override': {'_target_': 'lightning.Trainer', 'max_epochs': 1}, 'expands': {'ensembles': {'num_ensembles': 5, 'aggregation_method': 'mean'}, 'systematics': {'up_qcd': {'datamodule': None, 'datamodule_override': None, 'dataset': 'fair_universe_local', 'dataset_override': {'paths': '../DATA/fair-universe/data/*.parquet', 'features_

In [37]:
cfg_systematic = resolve_defaults(cfg.estimators.model_A.expands, node_type="systematics")
cfg_systematic.systematics.up_qcd

{'datamodule': None, 'datamodule_override': None, 'dataset': 'fair_universe_local', 'dataset_override': {'paths': '../DATA/fair-universe/data/*.parquet', 'features_columns': ['PRI_lep_pt', 'PRI_lep_eta', 'PRI_lep_phi', 'PRI_had_pt', 'PRI_had_eta', 'PRI_had_phi', 'PRI_met', 'PRI_met_phi'], 'labels_columns': ['PRI_n_jets'], 'format': 'automatic', 'dak_reader_kwargs': {}, 'max_number_events': 20000}, 'model': None, 'model_override': None, 'trainer': None, 'trainer_override': None}

In [48]:
systematic_merged = OmegaConf.merge(
    OmegaConf.to_container(cfg_systematic.systematics.up_qcd, resolve=False),
    cfg_resolved.estimators.model_A,
)

In [50]:
systematic_merged.datamodule_override

{'_target_': 'ml.lightning.data.padded_datamodule.PaddedDataModule', 'multiprocessing_type': 'torch', 'batch_size': 1024}

In [49]:
print(json.dumps(OmegaConf.to_container(systematic_merged), indent=4))

{
    "datamodule": "padded",
    "datamodule_override": {
        "_target_": "ml.lightning.data.padded_datamodule.PaddedDataModule",
        "multiprocessing_type": "torch",
        "batch_size": 1024
    },
    "dataset": "fair_universe",
    "dataset_override": {
        "paths": "/home/kylian/work/NEEDLE/FAIR_Universe_HiggsML_data/FAIR_Universe_HiggsML_data.parquet",
        "features_columns": [
            "PRI_lep_pt"
        ],
        "labels_columns": [
            "PRI_n_jets"
        ],
        "format": "automatic",
        "dak_reader_kwargs": {},
        "max_number_events": 100000
    },
    "model": "mock_transformer",
    "model_override": {
        "_target_": "ml.lightning.models.mock_transformer.MockTransformerModule",
        "factor": 0.1,
        "patience": 10,
        "init_lr": 0.001
    },
    "trainer": "default",
    "trainer_override": {
        "_target_": "lightning.Trainer",
        "max_epochs": 1
    },
    "expands": {
        "ensembles": {
      